# TEM Analyzer v2.3.0 — ABL 방식 직접 설치 (setup r2)
ABL 1.4에서 쓰던 **노트북 안의 직접 pip 설치 방식**을 사용합니다. 별도 설치 도우미 파일을 import하지 않습니다. 학습/데이터 분할/GT 변환은 실행하지 않습니다.

1. 기존 `TEM_Analyzer_v2` 프로그램 폴더에 이 노트북을 넣습니다.
2. ABL에서 사용하던 Python 3.11 이상/64-bit CUDA 커널을 선택합니다.
3. 첫 셀 → 설치 셀 → 환경 확인 → 모델 경로 → 시험/웹 서버 시작 순으로 실행합니다.

**설치 셀은 ABL처럼 기본 True입니다.** 최초 설치 후 `INSTALL_PACKAGES=False`로 두세요. 학습 중인 공유 환경에는 설치하지 마세요. 서버 시작/종료는 각각 명시적으로 True로 바꿀 때만 실행합니다.

이 노트북은 설치 부분이 자체 포함되어 있지만, Analyzer 실행에는 `run.py`, `tem_analyzer/`, `web/`, `adaptation_backend/`, requirements 등 실제 프로그램 소스가 필요합니다. 가중치는 자동 다운로드하지 않습니다. 회사 자료/토큰/노트북 출력은 공개 저장소에 올리지 마세요.


## 1. 경로 확인 — ABL과 동일한 기본 모듈만 사용
프로그램 폴더에서 열었다면 그대로 실행합니다. 찾지 못하면 `APP_DIR`에 압축을 푼 `TEM_Analyzer_v2`의 절대 경로를 넣습니다. 모델/프로젝트 폴더가 아니라 **프로그램 폴더**입니다.

`USE_OCR=False`는 설치/검사에서 OCR만 생략합니다. GPU가 없는 PC에서 시험할 때만 `REQUIRE_CUDA=False`로 설정합니다.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

APP_DIR = ""              # 예: "/workspace/TEM_Analyzer_v2"
REQUIRE_CUDA = True        # CPU 시험만 할 때 False
USE_OCR = True             # OCR 미사용 시 False

if APP_DIR:
    ROOT = Path(APP_DIR).expanduser().resolve()
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "tem_analyzer").is_dir():
        ROOT = ROOT.parent
if not (ROOT / "tem_analyzer").is_dir() or not (ROOT / "run.py").is_file():
    raise FileNotFoundError("APP_DIR에 TEM_Analyzer_v2 프로그램 폴더의 경로를 입력하세요.")
if not (ROOT / "requirements.txt").is_file():
    raise FileNotFoundError("requirements.txt가 없습니다. 전체 소스 ZIP을 압축 해제하세요.")
if sys.version_info < (3, 11):
    raise RuntimeError("Analyzer는 Python 3.11 이상이 필요합니다. 해당 커널을 선택하세요.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT)
print("Kernel Python:", sys.executable)


## 2. 패키지 직접 설치 — ABL 설치 셀에서 필요한 부분만 사용
ABL과 같이 `sys.executable -m pip`로 **현재 커널**에 설치하며 기존 torch/torchvision 버전을 고정합니다. 이미 설치된 주요 수치 패키지도 보존하고 충돌하면 중단합니다. ABL의 학습 패키지 대신 Analyzer requirements 및 선택적인 EasyOCR만 설치합니다. 앱은 동봉 SAM 소스를 직접 사용하므로 SAM용 별도 pip 설치가 필요 없습니다.

- 첫 설치는 True, 이후 False. 설치 후 커널을 재시작했다면 첫 셀 → 설치 False → 계속 진행합니다.
- 사내 미러는 ABL과 같은 `PIP_INDEX_URL`, CUDA 저장소는 `TORCH_INDEX_URL`, 오프라인은 `WHEELHOUSE`를 사용합니다. URL에 비밀번호/토큰을 넣지 마세요.
- 기존 CUDA torch 쌍이 정상 설치돼 있으면 저장소 설정을 바꿀 필요가 없습니다.
- torch가 누락된 새 환경은 승인된 CUDA 저장소 또는 서버 OS/Python/CUDA에 맞는 wheelhouse를 먼저 지정하세요. 드라이버/버전을 자동으로 추측하거나 보안 설정을 변경하지 않습니다.


In [ ]:
import importlib.metadata as metadata
import tempfile

INSTALL_PACKAGES = True  # ABL과 동일: 최초 설치 후 False
PIP_INDEX_URL = ""        # 사내 pip 설정이 있으면 빈 값 유지
TORCH_INDEX_URL = ""      # torch가 없을 때만 승인된 CUDA 패키지 저장소
WHEELHOUSE = ""           # 오프라인 wheel 폴더. 지정하면 --no-index 사용

if INSTALL_PACKAGES:
    def installed_version(name):
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            return None

    keep = ("torch", "torchvision", "numpy", "scipy", "Pillow", "matplotlib", "opencv-python-headless")
    pinned = {name: installed_version(name) for name in keep}
    missing_torch = not all(pinned[name] for name in ("torch", "torchvision"))
    if missing_torch and not (TORCH_INDEX_URL or WHEELHOUSE):
        raise RuntimeError("기존 ABL CUDA 커널을 선택하세요. 새 환경이면 TORCH_INDEX_URL 또는 WHEELHOUSE를 지정하세요.")
    if WHEELHOUSE and not Path(WHEELHOUSE).expanduser().is_dir():
        raise FileNotFoundError("WHEELHOUSE 경로를 확인하세요.")
    for index in (PIP_INDEX_URL, TORCH_INDEX_URL):
        if index and (not index.startswith("https://") or any(c in index for c in "@?#")):
            raise ValueError("저장소는 자격 증명 없는 HTTPS URL을 사용하세요.")
    print("보존할 패키지:", pinned)
    with tempfile.TemporaryDirectory(prefix="tem_abl_setup_") as td:
        constraints = Path(td) / "existing_versions.txt"
        def write_constraints():
            constraints.write_text("".join(f"{k}=={v}\n" for k, v in pinned.items() if v), encoding="utf8")
        write_constraints()
        def pip_install(args, index=""):
            cmd = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                   "--upgrade-strategy", "only-if-needed", "-c", str(constraints)]
            if WHEELHOUSE:
                cmd += ["--no-index", "--find-links", str(Path(WHEELHOUSE).expanduser().resolve())]
            elif index or PIP_INDEX_URL:
                cmd += ["--index-url", index or PIP_INDEX_URL]
            subprocess.run(cmd + args, cwd=ROOT, check=True)
        if missing_torch:
            pip_install(["torch", "torchvision"], index=TORCH_INDEX_URL)
            for name in ("torch", "torchvision"):
                pinned[name] = installed_version(name)
            if not all(pinned[name] for name in ("torch", "torchvision")):
                raise RuntimeError("PyTorch 설치 결과를 확인하세요.")
            write_constraints()  # OCR 설치가 새 CUDA 쌍도 교체하지 못하게 고정
        args = ["-r", str(ROOT / "requirements.txt")]
        if USE_OCR:
            args += ["-r", str(ROOT / "requirements-ocr.txt")]
        pip_install(args)
    subprocess.run([sys.executable, "-m", "pip", "check"], cwd=ROOT, check=True)
    print("설치 완료. 다음부터 INSTALL_PACKAGES=False로 두세요.")
else:
    print("패키지 설치 생략")


## 3. 실제 환경 점검
같은 Python의 **새 프로세스**에서 import, pip 충돌, CPU 및 CUDA의 tensor/NMS 연산을 검사합니다. 그러므로 설치 전 노트북에 남은 torch import 캐시로 결과를 판단하지 않습니다.

CUDA 검사가 실패하면 GPU 할당, 서버 드라이버, 선택한 커널, PyTorch 빌드를 확인하세요. 노트북은 드라이버를 설치하거나 권한을 변경하지 않습니다. 기존 ABL이 Python 3.10 환경이었다면 Analyzer용 Python 3.11 이상 환경이 필요합니다.


In [ ]:
# 설치 후 import 캐시와 모델 메모리를 분리하도록 같은 Python의 새 프로세스에서 검사합니다.
environment_code = "from pathlib import Path\nimport sys, subprocess, json\ndef environment_report(root, *, require_cuda=True, use_ocr=True):\n    root = Path(root).resolve()\n    import struct\n    if sys.version_info < (3, 11) or struct.calcsize('P') != 8:\n        raise RuntimeError('Python 3.11 이상 / 64-bit 커널 필요')\n    import fastapi, uvicorn, numpy, scipy, PIL, cv2, matplotlib, httpx  # noqa: F401\n    import torch, torchvision\n    try:\n        import python_multipart  # noqa: F401\n    except ImportError:\n        import multipart  # noqa: F401\n    if use_ocr:\n        import easyocr  # noqa: F401\n    sys.path.insert(0, str(root / 'adaptation_backend/vendor/segment-anything'))\n    from segment_anything import sam_model_registry  # noqa: F401\n    cuda = torch.cuda.is_available()\n    if require_cuda and not cuda:\n        raise RuntimeError('CUDA 사용 불가. GPU 할당/드라이버/torch 빌드/커널 확인. CPU 시험만 하려면 REQUIRE_CUDA=False.')\n    devices = ['cpu', 'cuda'] if require_cuda else ['cpu']\n    for device in devices:\n        boxes = torch.tensor([[0., 0., 2., 2.], [0., 0., 1., 1.]], device=device)\n        torchvision.ops.nms(boxes, torch.tensor([.9, .8], device=device), .5)\n        x = torch.ones((32, 32), device=device)\n        if not torch.isfinite(x @ x).all().item():\n            raise RuntimeError(f'{device} tensor 연산 실패')\n    if require_cuda:\n        torch.cuda.synchronize()\n    report = {'python': sys.executable, 'torch': torch.__version__, 'torchvision': torchvision.__version__,\n              'cuda_build': torch.version.cuda, 'cuda_available': cuda, 'tested_devices': devices,\n              'gpu': torch.cuda.get_device_name(0) if cuda else None,\n              'gpu_total_GiB': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda else None,\n              'ocr_device': 'cpu', 'masked_ecc_available': hasattr(cv2, 'findTransformECCWithMask')}\n    subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    return report\n"
environment_call = f"\nenvironment_report({str(ROOT)!r}, require_cuda={REQUIRE_CUDA!r}, use_ocr={USE_OCR!r})"
subprocess.run([sys.executable, "-u", "-c", environment_code + environment_call], cwd=ROOT, check=True)


## 4. ABL 1.4 파일을 어디에 넣나요?
```text
TEM_Analyzer_v2/
├── setup.ipynb
├── run.py
├── tem_analyzer/                 ← Analyzer 소스, ABL 소스로 덮어쓰지 않음
├── web/
├── adaptation_backend/          ← SAM/Refiner 소스 이미 포함
├── models/
│   ├── sam_vit_h_4b8939.pth      ← ABL의 weights/에서 복사 (필수)
│   ├── abl14/
│   │   └── decoder_abl/
│   │       ├── adaptation.pt    ← ABL export 결과 (선택)
│   │       ├── inference_settings.json  ← 기록용, 선택
│   │       └── model_manifest.json      ← 기록용, 선택
│   └── easyocr/
│       ├── craft_mlt_25k.pth    ← OCR 사용 시 필수, ABL에는 없음
│       └── english_g2.pth      ← OCR 사용 시 필수, ABL에는 없음
└── projects/
    └── company_session/        ← Analyzer가 저장하는 작업 프로젝트
```

- **기본 SAM만 실행**하면 `ADAPTATION=""`. ABL 학습 결과까지 쓰려면 export한 `adaptation.pt` 경로를 지정합니다. Decoder/Refiner 실험은 폴더를 나누고 한 번에 하나를 선택합니다.
- `02_evaluate_export.ipynb`의 실제 출력 경로에서 가져옵니다. `best.pt`도 schema=1 bundle이면 로드할 수 있지만 export 파일이 우선입니다.
- 기본 SAM은 **학습에 사용한 바로 그 파일**이어야 합니다. 모델 종류/SHA256/전처리/state 호환성은 다음 실제 로드에서 검사합니다. 학습 결과 하나만 복사해서는 안 됩니다.
- ABL의 `.venv`, `temlab`, `vendor`, `requirements.txt`, config, 학습 데이터, embedding cache는 옮기지 않아도 됩니다.
- 이미 서버에 파일이 있으면 아래 경로에 절대 경로를 지정해 공간을 절약할 수 있습니다. 자동 복사/덮어쓰기/다운로드는 하지 않습니다.
- OCR 두 파일은 이전 Analyzer에서 복사하거나 [검출 ZIP](https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip), [영어 ZIP](https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip)을 승인된 방법으로 받고 **압축을 풀어** 배치합니다.


In [ ]:
# 파일 검사도 노트북 내부에서 수행합니다. 파일을 복사/다운로드/덮어쓰지 않습니다.
SAM_NAMES = {"vit_h": "sam_vit_h_4b8939.pth", "vit_l": "sam_vit_l_0b3195.pth", "vit_b": "sam_vit_b_01ec64.pth"}
def model_paths(root, variant='vit_h', checkpoint='', adaptation='', ocr_dir='models/easyocr', use_ocr=True):
    root = Path(root).resolve()
    if variant not in SAM_NAMES:
        raise ValueError('SAM1 vit_h/vit_l/vit_b만 지원합니다. SAM2 가중치는 호환되지 않습니다.')
    def resolve(value):
        p = Path(value).expanduser()
        return (root / p).resolve() if not p.is_absolute() else p.resolve()
    base = resolve(checkpoint or f'models/{SAM_NAMES[variant]}')
    adapted = resolve(adaptation) if adaptation else None
    ocr = resolve(ocr_dir)
    required = [base] + ([adapted] if adapted else [])
    if use_ocr:
        required += [ocr / 'craft_mlt_25k.pth', ocr / 'english_g2.pth']
    missing = [str(p) for p in required if not p.is_file() or p.stat().st_size == 0]
    if missing:
        raise FileNotFoundError('파일 없음/빈 파일 (자동 다운로드 안 함):\n' + '\n'.join(missing))
    return {'checkpoint': str(base), 'variant': variant, 'adaptation_path': str(adapted) if adapted else None,
            'ocr_dir': str(ocr)}


SAM_VARIANT = "vit_h"  # ABL 기본 모델. B를 썼다면 vit_b 및 파일명도 변경
SAM_CHECKPOINT = "models/sam_vit_h_4b8939.pth"
ADAPTATION = ""        # 선택: "models/abl14/decoder_abl/adaptation.pt"
OCR_DIR = "models/easyocr"
PROJECT_DIR = "projects/company_session"  # 기존 프로젝트는 전체 폴더의 절대 경로
PORT = 8765

FILES = model_paths(ROOT, SAM_VARIANT, SAM_CHECKPOINT, ADAPTATION, OCR_DIR, USE_OCR)
DEVICE = "cuda" if REQUIRE_CUDA else "cpu"
print(json.dumps(FILES, ensure_ascii=False, indent=2))
print("사용 장치:", DEVICE)
print("프로젝트:", PROJECT_DIR)


## 5. 실제 SAM/선택적 OCR 시험 — 회사 영상 없이 검사
로컬 가중치를 읽고 합성 이미지에서 두 번 추론합니다. 첫 추론은 이미지 인코딩 포함, 두 번째는 임베딩 재사용입니다. CUDA 시간은 동기화해서 측정합니다. 검사가 끝나면 별도 프로세스가 종료되어 **테스트 모델의 GPU 메모리를 반환**합니다.

이 시험은 실행/형식 검사이며 실제 TEM 경계 정확도 검증은 아닙니다. 학습 bundle의 mask가 합성 이미지에서 비어도 실행 검사와 회사 정확도를 구분해야 합니다. ABL 학습 시 AMP 설정과 Analyzer 추론의 정밀도 차이도 있어 수치가 완전히 같다고 단정하지 않습니다. OOM이면 다른 학습 프로세스의 메모리 사용을 확인하고 작업을 조정하세요. 다른 프로세스는 자동 종료하지 않습니다.


In [ ]:
RUN_SAM_TEST = True  # 첫 확인 후 False로 생략 가능
if RUN_SAM_TEST:
    smoke_code = "from pathlib import Path\nimport sys, time, json\ndef sam_smoke(root, checkpoint, variant, device, adaptation=None, ocr_dir=None):\n    \"\"\"Real local weights, synthetic pixels. Timings are NOT an accuracy score.\"\"\"\n    root = Path(root).resolve()\n    sys.path.insert(0, str(root))\n    import numpy as np\n    import torch\n    from tem_analyzer.sam_service import ModelService\n    model = ModelService()\n    start = time.perf_counter()\n    info = model.load(checkpoint, variant, device, adaptation_path=adaptation)\n    if device == 'cuda':\n        torch.cuda.synchronize()\n    load_seconds = time.perf_counter() - start\n    image = np.full((192, 256, 3), 30, dtype=np.uint8)\n    image[60:130, 24:232] = 180\n    timings = []\n    for _ in range(2):\n        if device == 'cuda':\n            torch.cuda.synchronize()\n        start = time.perf_counter()\n        result = model.prompt(image, [[128, 96, 1]], [24, 60, 232, 130])\n        if device == 'cuda':\n            torch.cuda.synchronize()\n        if result['mask'].shape != (192, 256) or not np.isfinite(result['probability']).all():\n            raise RuntimeError('SAM 출력 크기/수치 검사 실패')\n        timings.append({'seconds': round(time.perf_counter() - start, 3),\n                        'embedding_reused': result['embedding_reused'], 'mask_pixels': int(result['mask'].sum())})\n    if ocr_dir:\n        from tem_analyzer.ocr import read_words\n        read_words(image, ocr_dir, 'en')  # Loads + runs existing offline CPU OCR, no downloads.\n    report = {'device': info['device'], 'variant': variant, 'adapted': info['adapted'],\n              'load_seconds': round(load_seconds, 3), 'inference': timings,\n              'ocr_executed': bool(ocr_dir), 'scope': 'runtime check only; not TEM segmentation accuracy'}\n    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n    return report\n"
    smoke_args = [str(ROOT), FILES["checkpoint"], SAM_VARIANT, DEVICE,
                  FILES["adaptation_path"], FILES["ocr_dir"] if USE_OCR else None]
    smoke_call = "\nsam_smoke(" + ", ".join(repr(v) for v in smoke_args) + ")"
    subprocess.run([sys.executable, "-u", "-c", smoke_code + smoke_call], cwd=ROOT, check=True)
else:
    print("SAM 시험 생략")


## 6. 웹 서버 시작
`START_SERVER=True`로 바꾸고 실행합니다. 모델을 서버에서 자동 로드하려면 `LOAD_MODEL_ON_START=True`를 유지하세요. 첫 로드 시 가중치 해시 검사와 GPU 메모리 할당으로 시간이 걸릴 수 있습니다.

- 이미 이 노트북에서 실행한 서버는 재사용합니다. 다른 프로세스가 포트를 사용하면 멈추고 안내합니다.
- 프로젝트당 서버 1개만 사용하세요. 동일 프로젝트 동시 편집은 지원하지 않습니다.
- `127.0.0.1`만 바인딩하며 **인증 없는 공개 서비스로 노출하지 않습니다**.
- UI에서 모델을 직접 바꿀 때는 아래 출력한 경로/variant/device를 사용하세요. OCR 폴더는 UI의 **OCR 설정 → 로컬 EasyOCR 폴더**에 입력합니다.


In [ ]:
# 서버 실행/종료 보조 코드도 자체 포함합니다. 별도 tools 패키지를 import하지 않습니다.
import socket, time, tempfile
from urllib.error import HTTPError, URLError
from urllib.request import ProxyHandler, Request, build_opener

class NotebookServer:
    """Own-process-only lifecycle. No PID files, no killing unknown servers."""
    def __init__(self, root, project, port=8765):
        self.root = Path(root).resolve()
        project = Path(project).expanduser()
        self.project = (self.root / project).resolve() if not project.is_absolute() else project.resolve()
        if isinstance(port, bool) or not isinstance(port, int) or not 1024 <= port <= 65535:
            raise ValueError('PORT는 1024~65535 정수여야 합니다.')
        self.port = port
        self.url = f'http://127.0.0.1:{port}'
        self.process = None
        self.log_path = None

    def request(self, path, payload=None, timeout=10):
        if self.process is None or self.process.poll() is not None:
            raise RuntimeError('이 노트북에서 시작한 서버가 실행 중이 아닙니다.')
        request = Request(self.url + path, data=json.dumps(payload).encode() if payload is not None else None,
                          headers={'Content-Type': 'application/json'} if payload is not None else {})
        # Only loopback: never send local API requests via a corporate HTTP proxy.
        opener = build_opener(ProxyHandler({}))
        try:
            with opener.open(request, timeout=timeout) as response:
                return json.load(response)
        except HTTPError as exc:
            raise RuntimeError(f'HTTP {exc.code}: {exc.read(1000).decode("utf8", "replace")}') from exc

    def start(self, timeout=45):
        if self.process is not None and self.process.poll() is None:
            print('이미 실행 중:', self.url)
            return self.url
        with socket.socket() as sock:
            try:
                sock.bind(('127.0.0.1', self.port))
            except OSError as exc:
                raise RuntimeError(f'포트 {self.port} 사용 중. 기존 프로세스를 종료하지 않습니다. 다른 포트를 선택하세요.') from exc
        self.project.mkdir(parents=True, exist_ok=True)
        # Check actual writes, not os.access (unreliable on network mounts/ACLs).
        with tempfile.TemporaryFile(dir=self.project):
            pass
        logs = self.root / 'test-output/server-setup'
        logs.mkdir(parents=True, exist_ok=True)
        self.log_path = logs / f'server-{self.port}-{time.time_ns()}.log'
        env = os.environ.copy()
        env['PYTHONUNBUFFERED'] = '1'
        with self.log_path.open('wb') as log:
            self.process = subprocess.Popen([sys.executable, '-u', str(self.root / 'run.py'),
                                             '--port', str(self.port), '--project', str(self.project)],
                                            cwd=self.root, env=env, stdout=log, stderr=subprocess.STDOUT,
                                            creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
        deadline = time.monotonic() + timeout
        try:
            while time.monotonic() < deadline:
                if self.process.poll() is not None:
                    raise RuntimeError('서버 시작 실패')
                try:
                    state = self.request('/api/state', timeout=1)
                    if 'schema_version' in state and self.process.poll() is None:
                        print('실행 중:', self.url, '\n프로젝트:', self.project, '\n로그:', self.log_path)
                        return self.url
                except (URLError, TimeoutError, OSError):
                    pass
                time.sleep(.25)
            raise TimeoutError('웹 서버 응답 대기 시간 초과')
        except BaseException:
            self.stop()
            print(self.log_path.read_text(encoding='utf8', errors='replace')[-4000:])
            raise

    def stop(self):
        if self.process is not None and self.process.poll() is None:
            # Call only after UI jobs have completed and project saves are done.
            if os.name == 'nt':
                # A Windows venv python.exe may be a redirector with a child Python.
                # Terminating only the parent leaves uvicorn running and files locked.
                result = subprocess.run(['taskkill', '/PID', str(self.process.pid), '/T', '/F'],
                                        capture_output=True, text=True,
                                        creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
                if result.returncode and self.process.poll() is None:
                    print('소유한 서버 트리 종료 실패:', result.stderr.strip())
                    return False
            else:
                self.process.terminate()
            try:
                self.process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                print('아직 종료 중입니다. 강제 종료/프로젝트 삭제를 하지 않았습니다.')
                return False
        return True


START_SERVER = False       # 실행할 때 True
LOAD_MODEL_ON_START = True

if START_SERVER:
    wanted = NotebookServer(ROOT, PROJECT_DIR, PORT)
    previous = globals().get("TEM_SERVER")
    if previous is not None and previous.process is not None and previous.process.poll() is None:
        if (previous.root, previous.project, previous.port) != (wanted.root, wanted.project, wanted.port):
            raise RuntimeError("기존 서버가 실행 중입니다. 작업 저장 후 종료 셀을 실행하고 설정을 변경하세요.")
        TEM_SERVER = previous
    else:
        TEM_SERVER = wanted
    TEM_SERVER.start()
    payload = {k: FILES[k] for k in ("checkpoint", "variant", "adaptation_path")}
    payload["device"] = DEVICE
    if LOAD_MODEL_ON_START:
        current_model = TEM_SERVER.request("/api/state").get("model") or {}
        if any(current_model.get(k) != v for k, v in payload.items()):
            # A large ViT-H/model hash check can take time; request goes only to our loopback process.
            print(TEM_SERVER.request("/api/model/load", payload, timeout=900))
        else:
            print("같은 모델이 이미 로드되어 있습니다. 파일 자체를 교체했다면 UI에서 다시 로드하세요.")
    print("서버 주소:", TEM_SERVER.url)
    print("UI OCR 폴더:", FILES["ocr_dir"] if USE_OCR else "OCR 검사 생략 상태")
    print("아래 원격 접속 안내를 확인하세요.")
else:
    print("서버 미시작. START_SERVER=True로 바꿔 이 셀을 실행하세요.")


## 7. 내 PC에서 계측 UI 열기
서버의 localhost 주소는 PC의 localhost와 다릅니다. **회사에서 허용하는 포트 전달**을 사용하세요.

**SSH가 가능한 경우 — 아래 명령은 노트북이 아니라 내 PC 터미널에서 실행**
```text
ssh -N -L 8765:127.0.0.1:8765 USER@SERVER
```
계정/서버/포트는 실제 값으로 바꿉니다. 연결을 유지한 채 PC 브라우저에서 `http://127.0.0.1:8765/`를 엽니다. VS Code Remote-SSH의 Ports 탭에서 8765를 **Private**로 전달해도 됩니다.

**중요한 제약**
- 노트북 커널이 SSH 서버와 다른 계산 노드/컨테이너이면 위 명령만으로 연결되지 않습니다. IT가 허용한 노드 전달 경로가 필요합니다.
- 현재 앱의 `/api`·`/web`는 루트 절대 경로입니다. **일반 JupyterHub `/user/.../proxy/8765/` 링크만 붙이는 방식은 지원 완료 상태가 아닙니다.** SSH 없이 프록시만 가능한 서버라면 해당 URL 구조에 맞춘 추가 대응이 필요합니다.
- Jupyter 프록시를 설치할 때는 커널 환경이 아니라 Jupyter 서버 환경에 설치/활성화해야 합니다. 이 노트북은 프록시·방화벽·인증 설정을 변경하지 않습니다.
- 상시 서비스/다중 사용자 운영은 별도 설계가 필요합니다. Jupyter 세션/작업 스케줄러가 종료되면 웹 서버도 종료될 수 있습니다.


## 8. 로그 확인 (문제가 있을 때)
회사 경로/오류 정보가 포함될 수 있으므로 이 출력 전체를 공개 저장소에 업로드하지 마세요.


In [ ]:
server = globals().get("TEM_SERVER")
if server is not None and server.log_path is not None:
    print("PID:", server.process.pid, "exit code:", server.process.poll())
    print(server.log_path.read_text(encoding="utf8", errors="replace")[-6000:])
else:
    print("이 커널에서 시작한 서버가 없습니다.")


## 9. 작업 완료 후 종료
UI의 SAM/일괄/저장 작업이 모두 끝난 것을 확인한 후 `STOP_SERVER=True`로 바꿔 실행합니다. **Run All은 기본값 False이므로 방금 시작한 서버를 바로 종료하지 않습니다.** Windows에서는 terminate가 강제 종료이므로 저장 중 실행하면 안 됩니다.

이 노트북이 시작한 프로세스만 종료하며 프로젝트/모델 파일은 삭제하지 않습니다. 커널 재시작으로 참조가 사라진 서버는 PID를 추측해서 죽이지 않습니다. 포트 충돌 시 IT/서버 터미널에서 실행 기록을 확인하세요.


In [ ]:
STOP_SERVER = False  # 저장 완료 후 True
if STOP_SERVER:
    server = globals().get("TEM_SERVER")
    if server is None:
        print("현재 커널에는 소유한 서버 참조가 없습니다. 다른 프로세스를 종료하지 않습니다.")
    elif server.stop():
        print("웹 서버 종료 완료. 프로젝트와 모델 파일은 보존됩니다.")
else:
    print("서버 종료 생략")


## GPU로 얼마나 빨라지나요?
- **GPU 대상:** SAM 이미지 인코딩, 프롬프트 추론, 학습 Decoder/Refiner.
- **현재 CPU:** EasyOCR(`gpu=False`), Canny/Sobel 등 전처리, Gradient/DP, 회전, 두께/CD 계측.
- 일괄 SAM은 이미지마다 순차 처리합니다. 자동 다중 GPU 분산은 아닙니다.
- GPU 모델/VRAM/점 수/이미지/다른 학습 작업에 따라 달라 **속도 배수는 회사 서버 실측 전 보장하지 않습니다.**
- 현재 소스는 ViT-H decoder를 ViT-B에 억지로 붙이지 않고 기본 모델 SHA256/variant 불일치 시 중단합니다.

참고: [PyTorch 설치 및 CUDA 확인](https://pytorch.org/get-started/locally/), [공식 SAM](https://github.com/facebookresearch/segment-anything), [Jupyter 프록시 설치](https://jupyter-server-proxy.readthedocs.io/en/latest/install.html).

**다음 실행:** 설치 False → 환경/경로 확인 → 필요하면 SAM 시험 생략 → 서버 시작. 매번 재설치/재다운로드할 필요는 없습니다.
